# High-Utility Itemset Mining on Real Retail Baskets

**DMBI CA3 Content Beyond Syllabus, Division D15A, Group 09**

Jaiprakash Talreja (54), Adarsh Singh (52), Vidushi Singh (53), Piyush Shukla (49)

Module V teaches Apriori, which asks how *often* items are bought together. High-utility itemset mining asks how much those items are *worth*. This notebook implements HUI-Miner (Liu and Qu, CIKM 2012), checks it against brute force, and compares it with frequent itemset mining on the UCI Online Retail data.

**Main reference.** Gan, W., Lin, J. C.-W., Fournier-Viger, P., Chao, H.-C., Tseng, V. S. and Yu, P. S. (2021), *A Survey of Utility-Oriented Pattern Mining*, IEEE TKDE 33(4), 1306-1327. https://doi.org/10.1109/TKDE.2019.2942594

**Algorithm.** Liu, M. and Qu, J. (2012), *Mining High Utility Itemsets without Candidate Generation*, CIKM 2012, 55-64. https://doi.org/10.1145/2396761.2396773. The pair check is the EUCS idea from Fournier-Viger, P., Wu, C.-W., Zida, S. and Tseng, V. S. (2014), *FHM: Faster High-Utility Itemset Mining Using Estimated Utility Co-occurrence Pruning*, ISMIS 2014, 83-92. https://doi.org/10.1007/978-3-319-08326-1_9

**Data.** *Online Retail* (2015), UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

In [ ]:
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import json
import random
import time
import urllib.request

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori
from scipy import sparse

ROOT = Path.cwd()
DATA, FIG, RES = ROOT / "data", ROOT / "figures", ROOT / "results"
for folder in (DATA, FIG, RES):
    folder.mkdir(exist_ok=True)
M = {}

BLUE, ORANGE, INK2, GRID, AXIS, SURFACE = "#2a78d6", "#eb6834", "#52514e", "#e1e0d9", "#c3c2b7", "#fcfcfb"
mpl.rcParams.update({
    "figure.dpi": 80, "savefig.dpi": 200, "font.family": "Segoe UI", "font.size": 10, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlesize": 12, "axes.titleweight": "semibold",
    "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False, "grid.color": GRID, "grid.linewidth": .7,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
})

## 1. HUI-Miner

A utility-list holds, for one itemset, a row per transaction: `(tid, iutil, rutil)`. `iutil` is the utility of the itemset there and `rutil` is the utility of everything later in the processing order. If the sum of `iutil + rutil` is below the threshold, no extension of the itemset can reach it, so the whole branch is skipped.

These baskets are large, so nearly every item passes the first TWU filter and plain HUI-Miner would try millions of joins. The implementation therefore adds the pair check from FHM: a pair whose TWU misses the threshold is skipped without a join. Both versions are checked against exhaustive search below.

In [ ]:
def hui_miner(transactions, minutil, use_pair_check=True):
    """Return {itemset: utility} for every itemset with utility >= minutil, plus search statistics."""
    totals = [sum(t.values()) for t in transactions]
    twu = Counter()
    for t, total in zip(transactions, totals):
        for item in t:
            twu[item] += total
    keep = [i for i in twu if twu[i] >= minutil]
    rank = {item: r for r, item in enumerate(sorted(keep, key=lambda i: (twu[i], str(i))))}

    # pair check: a pair whose TWU misses the threshold cannot be in any high-utility itemset
    allowed = None
    if use_pair_check and rank:
        cells = [(tid, rank[i]) for tid, t in enumerate(transactions) for i in t if i in rank]
        tids, ranks = map(np.array, zip(*cells))
        incidence = sparse.csr_matrix((np.ones(len(tids)), (tids, ranks)), shape=(len(transactions), len(rank)))
        pair = (incidence.T @ sparse.diags(totals) @ incidence).tocoo()
        mask = (pair.row < pair.col) & (pair.data >= minutil)
        allowed = set(zip(pair.row[mask].tolist(), pair.col[mask].tolist()))

    rows = defaultdict(lambda: ([], [], []))
    for tid, t in enumerate(transactions):
        rem = 0.0
        for item in sorted((i for i in t if i in rank), key=rank.get, reverse=True):
            ids, ius, rus = rows[rank[item]]
            ids.append(tid)
            ius.append(t[item])
            rus.append(rem)
            rem += t[item]
    lists = {r: tuple(np.array(column) for column in columns) for r, columns in rows.items()}
    stats = {"items_total": len(twu), "items_after_twu": len(keep), "pair_candidates": len(allowed) if allowed is not None else None,
             "lists_built": len(keep), "pairs_skipped": 0, "branches_pruned": 0}
    names_by_rank = {r: item for item, r in rank.items()}
    found = {}

    def join(prefix, px, py):
        common, ix, iy = np.intersect1d(px[0], py[0], assume_unique=True, return_indices=True)
        if len(common) == 0:
            return None
        iu = px[1][ix] + py[1][iy]
        if prefix is not None:
            iu = iu - prefix[1][np.searchsorted(prefix[0], common)]
        return common, iu, py[2][iy]

    def search(prefix_items, prefix_list, extensions):
        for pos, (r, ul) in enumerate(extensions):
            iutil, remain = float(ul[1].sum()), float(ul[2].sum())
            itemset = prefix_items + (r,)
            if iutil >= minutil:
                found[frozenset(names_by_rank[i] for i in itemset)] = iutil
            if iutil + remain >= minutil:
                next_ext = []
                for other, other_ul in extensions[pos + 1:]:
                    if allowed is not None and (r, other) not in allowed:
                        stats["pairs_skipped"] += 1
                        continue
                    joined = join(prefix_list, ul, other_ul)
                    stats["lists_built"] += 1
                    if joined is not None:
                        next_ext.append((other, joined))
                if next_ext:
                    search(itemset, ul, next_ext)
            else:
                stats["branches_pruned"] += 1

    search((), None, [(r, lists[r]) for r in sorted(lists)])
    return found, stats


def utility_of(itemset, transactions):
    return sum(sum(t[i] for i in itemset) for t in transactions if set(itemset) <= set(t))


def brute_force(transactions, minutil):
    items = sorted({i for t in transactions for i in t})
    found = {}
    for size in range(1, len(items) + 1):
        for combo in combinations(items, size):
            u = utility_of(combo, transactions)
            if u >= minutil:
                found[frozenset(combo)] = u
    return found


random.seed(7)
checks = 0
for trial in range(300):
    n_items = random.randint(3, 8)
    db = [{i: random.randint(1, 9) for i in random.sample(range(n_items), random.randint(1, n_items))} for _ in range(random.randint(2, 12))]
    threshold = random.randint(1, max(1, sum(sum(t.values()) for t in db) // 2))
    exact = brute_force(db, threshold)
    assert hui_miner(db, threshold)[0] == exact, (db, threshold)
    assert hui_miner(db, threshold, use_pair_check=False)[0] == exact, (db, threshold)
    checks += 1
M["verification"] = {"random_databases_matched_brute_force": checks}
print("HUI-Miner matched exhaustive search on", checks, "random databases")

### Worked example used on the slides

A six-basket toy shop. It shows the two ideas that make this topic harder than Apriori: frequency and value disagree, and a low-utility itemset can have a high-utility superset.

In [ ]:
PRICE = {"Bread": 1, "Milk": 1, "Eggs": 2, "Wine": 15, "Cheese": 7}
BASKETS = [{"Bread": 2, "Milk": 1}, {"Bread": 1, "Milk": 2, "Eggs": 1}, {"Bread": 1, "Eggs": 2}, {"Bread": 1, "Milk": 1}, {"Wine": 2, "Cheese": 1}, {"Wine": 1, "Cheese": 2, "Bread": 1}]
toy = [{i: q * PRICE[i] for i, q in b.items()} for b in BASKETS]
toy_min = 30
toy_found, toy_stats = hui_miner(toy, toy_min)


def row(items):
    return {"itemset": " + ".join(sorted(items)), "support": sum(set(items) <= set(t) for t in toy), "utility": utility_of(items, toy)}


M["toy"] = {
    "prices": PRICE, "baskets": BASKETS, "minutil": toy_min,
    "bread_milk": row(["Bread", "Milk"]), "wine": row(["Wine"]), "cheese": row(["Cheese"]), "wine_cheese": row(["Wine", "Cheese"]),
    "high_utility": sorted(([" + ".join(sorted(k)), v] for k, v in toy_found.items()), key=lambda x: -x[1]),
    "stats": toy_stats,
}
print(json.dumps({k: v for k, v in M["toy"].items() if k not in ("prices", "baskets")}, indent=1))

## 2. Real data: UCI Online Retail

12 months of transactions from a UK online gift retailer. Profit is not in the data, so utility here is revenue: quantity × unit price. That is the external utility used throughout, and it is stated wherever a result appears.

A cancellation appears as a separate negative row while the original sale stays in the file, so cancelled sales are removed as matched pairs. Skipping this step would make two cancelled bulk orders look like the most valuable sales in the file.

In [ ]:
XLSX = DATA / "OnlineRetail.xlsx"
CACHE = DATA / "online_retail.csv"
if not XLSX.exists():
    urllib.request.urlretrieve("https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx", XLSX)
if not CACHE.exists():
    pd.read_excel(XLSX).to_csv(CACHE, index=False)
raw = pd.read_csv(CACHE, dtype={"InvoiceNo": str, "StockCode": str}, low_memory=False)

df = raw.dropna(subset=["Description"]).copy()
df["StockCode"] = df["StockCode"].str.upper().str.strip()
cancel_rows = df["InvoiceNo"].str.startswith("C")
cancels = df[cancel_rows]
sales = df[~cancel_rows & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()

key = ["CustomerID", "StockCode", "Quantity"]
to_remove = cancels.assign(Quantity=-cancels["Quantity"]).groupby(key, dropna=True).size().rename("n").reset_index()
sales["occurrence"] = sales.groupby(key, dropna=False).cumcount()
sales = sales.merge(to_remove, on=key, how="left")
removed_pairs = int((sales["occurrence"] < sales["n"].fillna(0)).sum())
sales = sales[sales["occurrence"] >= sales["n"].fillna(0)].drop(columns=["occurrence", "n"])

non_product = sales["StockCode"].str.fullmatch(r"(POST|DOT|M|C2|BANK CHARGES|AMAZONFEE|S|D|CRUK|PADS|B|GIFT_.*|DCGS.*)")
sales = sales[~non_product].copy()
sales["revenue"] = sales["Quantity"] * sales["UnitPrice"]
names = sales.groupby("StockCode")["Description"].agg(lambda s: s.mode().iat[0]).str.strip().str.title().to_dict()

basket = sales.groupby(["InvoiceNo", "StockCode"])["revenue"].sum().round(2).reset_index()
transactions = [dict(zip(g["StockCode"], g["revenue"].astype(float))) for _, g in basket.groupby("InvoiceNo")]
total_rev = float(basket["revenue"].sum())

M["data"] = {
    "raw_rows": len(raw), "rows_clean": len(sales), "invoices": len(transactions), "items": int(basket["StockCode"].nunique()), "revenue_total": round(total_rev),
    "cancelled_pairs_removed": removed_pairs, "avg_items_per_basket": round(float(np.mean([len(t) for t in transactions])), 1),
    "date_from": str(pd.to_datetime(sales["InvoiceDate"]).min().date()), "date_to": str(pd.to_datetime(sales["InvoiceDate"]).max().date()),
}
print(M["data"])

## 3. Frequency against value

Both methods are given the same budget: the 20 best pairs and larger itemsets each can name.

In [ ]:
MIN_SUP = .01
top_codes = basket["StockCode"].value_counts()
top_codes = top_codes[top_codes / len(transactions) >= MIN_SUP].index
onehot = pd.DataFrame(False, index=range(len(transactions)), columns=list(top_codes))
position = {c: i for i, c in enumerate(onehot.columns)}
matrix = np.zeros(onehot.shape, dtype=bool)
for tid, t in enumerate(transactions):
    for code in t:
        if code in position:
            matrix[tid, position[code]] = True
onehot = pd.DataFrame(matrix, columns=onehot.columns)
start = time.time()
freq = apriori(onehot, min_support=MIN_SUP, use_colnames=True, max_len=4, low_memory=True)
apriori_seconds = time.time() - start
freq = freq[freq["itemsets"].map(len) >= 2].copy()


def itemset_stats(items):
    items = set(items)
    support, utility = 0, 0.0
    for t in transactions:
        if items <= t.keys():
            support += 1
            utility += sum(t[i] for i in items)
    return support, utility


freq["support_count"], freq["utility"] = zip(*(itemset_stats(s) for s in freq["itemsets"]))
top_freq = freq.sort_values("support_count", ascending=False).head(20)

MIN_UTIL_PCT = .1
MINUTIL = round(total_rev * MIN_UTIL_PCT / 100)
start = time.time()
huis, stats = hui_miner(transactions, MINUTIL)
hui_seconds = time.time() - start
hui_rows = pd.DataFrame([{"itemsets": k, "utility": v} for k, v in huis.items()])
hui_multi = hui_rows[hui_rows["itemsets"].map(len) >= 2].copy()
hui_multi["support_count"] = [itemset_stats(s)[0] for s in hui_multi["itemsets"]]
top_util = hui_multi.sort_values("utility", ascending=False).head(20)

freq_keys, util_keys = set(top_freq["itemsets"]), set(top_util["itemsets"])
overlap = len(freq_keys & util_keys)
missed_by_support = hui_multi[hui_multi["support_count"] / len(transactions) < MIN_SUP]
top_util_rare = top_util[top_util["support_count"] / len(transactions) < MIN_SUP]


def show(items):
    return " + ".join(sorted(names.get(i, i) for i in items))


M["compare"] = {
    "min_support_pct": MIN_SUP * 100, "minutil": MINUTIL, "minutil_pct_of_revenue": MIN_UTIL_PCT, "frequent_itemsets_size2plus": len(freq), "high_utility_total": len(huis), "high_utility_size2plus": len(hui_multi),
    "top20_overlap": overlap, "top_freq_utility_total": round(float(top_freq["utility"].sum())), "top_util_utility_total": round(float(top_util["utility"].sum())),
    "utility_gain_ratio": round(float(top_util["utility"].sum() / top_freq["utility"].sum()), 2), "missed_by_support": len(missed_by_support), "missed_pct": round(len(missed_by_support) / max(len(hui_multi), 1) * 100, 1),
    "top_util_rare_in_top20": len(top_util_rare), "apriori_seconds": round(apriori_seconds, 2), "hui_seconds": round(hui_seconds, 2),
    "top_freq": [{"itemset": show(r["itemsets"]), "support": int(r["support_count"]), "utility": round(float(r["utility"]))} for _, r in top_freq.head(5).iterrows()],
    "top_util": [{"itemset": show(r["itemsets"]), "support": int(r["support_count"]), "utility": round(float(r["utility"]))} for _, r in top_util.head(5).iterrows()],
    "search": stats,
}
print(json.dumps(M["compare"], indent=1))

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.6))
a = freq.sample(min(len(freq), 3000), random_state=1)
ax.scatter(a["support_count"], a["utility"], s=6, color=AXIS, linewidths=0, label="Frequent itemsets, support at least 1%")
ax.scatter(top_freq["support_count"], top_freq["utility"], s=42, color=BLUE, edgecolor=SURFACE, linewidths=.6, label="Top 20 by support")
ax.scatter(top_util["support_count"], top_util["utility"], s=42, color=ORANGE, edgecolor=SURFACE, linewidths=.6, label="Top 20 by utility")
ax.set(xscale="log", yscale="log", xlabel="Baskets containing the itemset (log scale)", ylabel="Revenue from the itemset, GBP (log scale)", title="Frequency and value pick different itemsets")
ax.grid(True)
ax.set_axisbelow(True)
ax.legend(loc="lower right", fontsize=8.5)
fig.savefig(FIG / "ca3_frequency_vs_value.png", bbox_inches="tight")
plt.show()

## 4. How the search narrows

Each stage of pruning is counted exactly: items that pass TWU, pairs that pass the TWU check, utility-lists actually built at every level, and the high-utility itemsets finally found. The counts are search effort, not seconds, because this is a teaching implementation in Python.

In [ ]:
effort = []
for pct in (1.0, .5, .25, MIN_UTIL_PCT):
    threshold = round(total_rev * pct / 100)
    t0 = time.time()
    found, st = hui_miner(transactions, threshold)
    effort.append({"minutil_pct": pct, "minutil": threshold, "high_utility_itemsets": len(found), "items_total": st["items_total"], "items_after_twu": st["items_after_twu"], "pair_candidates": st["pair_candidates"],
                   "lists_built": st["lists_built"], "pairs_skipped": st["pairs_skipped"], "branches_pruned": st["branches_pruned"], "seconds": round(time.time() - t0, 2)})
    print(effort[-1])
effort = pd.DataFrame(effort)
effort.to_csv(RES / "search_effort.csv", index=False)
M["effort"] = effort.to_dict("records")
M["all_possible_pairs"] = M["data"]["items"] * (M["data"]["items"] - 1) // 2

fig, ax = plt.subplots(figsize=(7.4, 3.9))
x = np.arange(len(effort))
width = .27
ax.bar(x - width, effort["pair_candidates"], width=width, color=BLUE, edgecolor=SURFACE, label="Pairs passing the TWU check")
ax.bar(x, effort["lists_built"], width=width, color=ORANGE, edgecolor=SURFACE, label="Utility-lists built, all levels")
ax.bar(x + width, effort["high_utility_itemsets"].clip(lower=1), width=width, color=AXIS, edgecolor=SURFACE, label="High-utility itemsets found")
ax.set_yscale("log")
ax.set_xticks(x, [f"{p:g}%" for p in effort["minutil_pct"]])
ax.set(title="Search effort as the threshold falls", xlabel="Minimum utility, % of total revenue", ylabel="Count (log scale)")
ax.grid(axis="y")
ax.set_axisbelow(True)
ax.legend(fontsize=8.5)
fig.savefig(FIG / "ca3_search_effort.png", bbox_inches="tight")
plt.show()

(RES / "metrics.json").write_text(json.dumps(M, indent=2, default=str), encoding="utf-8")
print("saved")